# AT-TPC Latent Space Exploration
---
This notebook explores AT-TPC event embeddings with k-means, PCA projection, PCA variance analysis, UMAP, t-SNE.

In [ ]:
import os
import sys
import numpy as np
sys.path.append('../../')
from clustering import t_SNE_clustering
from clustering import UMAP_embedding
from clustering import k_means_clustering
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

In [ ]:
# create a folder for results of global feature exploration

folder_path = './plots/exploration'
if not os.path.exists(folder_path):
  os.makedirs(folder_path, exist_ok=True)

## 1. Load Data
---
Set the feature and label paths, then load the `.npy` files.

**Labels must be verified reference taxonomy** (simulation truth, manual review, or another trusted source). Do not use the embedding model's own predictions as labels for supervised checks such as linear probing or k-means comparison.

**Partially labeled data:** mark unlabeled rows with `-1` (default). Label-free sections (PCA variance) use all events; label-colored plots show unlabeled points in gray; supervised sections require at least one labeled row.

**Fully unlabeled data:** set `labels_path = None`. Only PCA variance will run; skip k-means, PCA projection, UMAP, t-SNE, and cluster verification.

| Notebook section | Labels required? |
| --- | --- |
| PCA variance (3b) | No |
| k-means, PCA projection, UMAP, t-SNE | Optional (unlabeled shown in gray) |
| Cluster verification (4) | Yes (labeled rows only) |

By default, labels are used exactly as they appear in the file. Plot legends use the label values from the file, such as `0`, `1`, `2` or text values like `cat` and `dog`, unless you provide custom names.

**`label_groups`** (optional): combine raw label values into fewer classes. Set to `None` to skip regrouping. Each inner list becomes one class. Works with numeric or text labels.

**`class_names`** (optional): custom legend names. Leave as `None` to use the label values from the file. If provided, include one name per unique label after any regrouping.

**`unlabeled_values`** (optional): label values that mean "no verified class" (default: `-1`).



In [ ]:
# Set file paths
features_path = '../../data/features.npy' # CHANGE THE NAME OF THE DATA FILE AS NEEDED
labels_path = '../../data/master_labels.npy' # Set to None for fully unlabeled data

# Values that mark rows without a verified class label.
unlabeled_values = (-1,)

# Optional: regroup raw label values into fewer classes.
# None = use labels from the file as-is (default).
label_groups = None
# Numeric example:
# label_groups = [
#     [0, 1, 2],
#     [3],
#     [4, 5],
# ]
# Text example:
# label_groups = [
#     ["cat"],
#     ["dog"],
# ]

# Optional display names for plot legends.
# None = use label values from the file, such as "0", "1", "2" or "cat", "dog".
class_names = None
# class_names = ["Class 0", "Class 1", "Class 2"]

features = np.load(features_path)
print(f"Features loaded successfully! Shape: {features.shape}")

In [ ]:
# Load labels
label_data = np.load(labels_path)

In [ ]:
from label_utils import print_label_coverage, summarize_label_coverage

def load_labels(label_data):
    labels = np.asarray(label_data).ravel()
    if np.issubdtype(labels.dtype, np.floating):
        return labels
    if np.issubdtype(labels.dtype, np.number):
        return labels.astype(int)
    return labels.astype(str)

def apply_label_groups(labels, groups):
    if groups is None:
        return labels
    mapped = labels.copy()
    for new_label, raw_values in enumerate(groups):
        mapped[np.isin(labels, raw_values)] = new_label
    return mapped

labels = load_labels(label_data)
labels = apply_label_groups(labels, label_groups)

label_coverage = summarize_label_coverage(labels, unlabeled_values)
print_label_coverage(label_coverage)
has_labeled_data = label_coverage["n_labeled"] > 0

if has_labeled_data:
    labeled_unique = np.unique(labels[label_coverage["labeled_mask"]])
    if class_names is not None and len(class_names) != len(labeled_unique):
        raise ValueError(
            f"class_names must have {len(labeled_unique)} entries, got {len(class_names)}"
        )
    print("Labeled classes:", labeled_unique)
    print(
        "Labeled counts:",
        {str(v): int(np.sum(labels == v)) for v in labeled_unique},
    )
else:
    print("No verified labels found. Only label-free notebook sections can run.")

features = StandardScaler().fit_transform(features)
print(f"Data scaled successfully! {features.shape}")

## 2. K-Means
---
Run k-means on the embeddings, seperating them into k clusters. The clusters can then be compared with the grouped labels, using `class_names` for legend text when it is provided.

In [ ]:
if not has_labeled_data:
    raise RuntimeError("k-means comparison requires at least one verified labeled row.")

save_dir = './plots/k_means'

features_glob, cluster_labels, cluster_indices = k_means_clustering(
    features,
    labels,
    dimension=2,
    save_dir=save_dir,
    num_samples_to_print=10,
    class_names=class_names,
    unlabeled_values=unlabeled_values,
)

## 3. PCA Projection
---
Use PCA as a baseline before UMAP and t-SNE. PCA can project the scaled embeddings into either 2D or 3D by changing `dimension`. It shows the main directions of variation and uses the same labels and optional `class_names` as the other plots.

In [ ]:
if not has_labeled_data:
    raise RuntimeError("PCA projection with label coloring requires at least one verified labeled row.")

from clustering import pca_clustering

save_dir = './plots/pca'

# Use dimension=2 for a 2D plot or dimension=3 for a 3D plot.
pca_results = pca_clustering(
    features,
    labels,
    dimension=2,
    save_dir=save_dir,
    class_names=class_names,
    unlabeled_values=unlabeled_values,
)

## 3b. PCA Variance Analysis
---
Fit PCA on all embedding dimensions and report how many principal components are needed to reach a target explained-variance threshold (default 95%). The cumulative variance plot is saved under `./plots/pca/variance/`.

In [ ]:
from clustering import pca_variance_analysis

variance_threshold = 0.95

variance_results = pca_variance_analysis(
    features,
    variance_threshold=variance_threshold,
    save_dir="./plots/pca",
    plot_name="exploration",
)

variance_results["n_components"]

## 4. Check Cluster Labels
---
Print the verified labels for sampled events in each cluster. Requires labeled data and a prior k-means run.

In [ ]:
def true_labels(indices, labels):
    c_labels = []
    for index in indices:
        c_labels.append(labels[index])

    return c_labels

print("True labels verification per cluster:")
for i, cluster in enumerate(cluster_indices):
    print(f"Cluster {i}: {true_labels(cluster, labels)}")

## 5. UMAP Visualization
---
Run UMAP on the embeddings and color the 2D plot by verified label. Unlabeled rows appear in gray. Requires at least one labeled row for the legend; UMAP itself uses all events.

In [ ]:
neighbors = 50
fig, ax = plt.subplots(figsize=(10, 7))

umap_results = UMAP_embedding(
    features,
    dimension=2,
    ax=ax,
    labels=labels,
    neighbors=neighbors,
    class_names=class_names,
    save_dir='./plots'
)

ax.legend()
plt.title(f"UMAP 2D Manifold Embedding (n_neighbors={neighbors})")
plt.savefig('./plots/umap/umap_unified_manifold.png', dpi=200)
plt.show()

## 6. t-SNE Visualization
---
Run t-SNE on the embeddings and color the 2D plot by verified label. Unlabeled rows appear in gray. Requires at least one labeled row for the legend; t-SNE itself uses all events. The main hyperparameter to change is `perplexity`. The plot is saved under `./plots/t-sne/`.

In [ ]:
if not has_labeled_data:
    raise RuntimeError("t-SNE label coloring requires at least one verified labeled row.")

perplexity = 40
fig, ax = plt.subplots(figsize=(10, 7))

tsne_results = t_SNE_clustering(
    features,
    dimension=2,
    ax=ax,
    labels=labels,
    perplexity=perplexity,
    class_names=class_names,
    save_dir='./plots',
    unlabeled_values=unlabeled_values,
)

ax.legend()
plt.title(f"t-SNE 2D Manifold Embedding (perplexity={perplexity})")
plt.savefig('./plots/t-sne/tsne_unified_manifold.png', dpi=200)
plt.show()